<a href="https://colab.research.google.com/github/vladmsnk/dl_aith/blob/dl_hw3/dl_hw3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
%%capture
!pip install torchmetrics

In [9]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from tqdm.notebook import tqdm
from collections import defaultdict

from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


from IPython.display import clear_output
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

from torchmetrics import F1Score



import nltk
from nltk.probability import FreqDist
from nltk.tokenize import word_tokenize

In [10]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [11]:

df = pd.read_csv('trip-advisor-hotel-reviews.zip')

In [12]:
df.head()

,Review,Rating
0,nice hotel expensive parking got good deal sta...,4
1,ok nothing special charge diamond member hilto...,2
2,nice rooms not 4* experience hotel monaco seat...,3
3,"unique, great stay, wonderful time hotel monac...",5
4,"great stay great stay, went seahawk game aweso...",5


In [13]:
X = df['Review']
y = df['Rating']

In [14]:
X.head()

,Review
0,nice hotel expensive parking got good deal sta...
1,ok nothing special charge diamond member hilto...
2,nice rooms not 4* experience hotel monaco seat...
3,"unique, great stay, wonderful time hotel monac..."
4,"great stay great stay, went seahawk game aweso..."


In [15]:
y.head()

,Rating
0,4
1,2
2,3
3,5
4,5


In [16]:
import torch
import gc

torch.cuda.empty_cache()
gc.collect()


31

In [17]:
unique_ratings = y.unique()
print("Уникальные рейтинги:", unique_ratings)

Уникальные рейтинги: [4 2 3 5 1]


In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

In [19]:
train_corpus = list(X_train)

In [20]:
tokens = []

for text in tqdm(train_corpus):
  tokens.extend(word_tokenize(text))

tokens_filtered = [word for word in tokens if word.isalnum()]

  0%|          | 0/12294 [00:00<?, ?it/s]

In [21]:
max_words = 10000
dist = FreqDist(tokens_filtered)
tokens_filtered_top = [pair[0] for pair in dist.most_common(max_words-1)]

In [22]:
tokens_filtered_top[:10]

['hotel',
 'room',
 'not',
 'great',
 'good',
 'staff',
 'stay',
 'did',
 'just',
 'nice']

In [23]:
vocabulary = {v: k for k, v in dict(enumerate(tokens_filtered_top, 1)).items()}

In [24]:
len(vocabulary)

9999

In [25]:
def text_to_sequence(text, maxlen):
    result = []
    tokens = word_tokenize(text.lower())
    tokens_filtered = [word for word in tokens if word.isalnum()]
    for word in tokens_filtered:
        if word in vocabulary:
            result.append(vocabulary[word])
    padding = [0]*(maxlen-len(result))
    return padding + result[-maxlen:]

In [26]:
%%capture
!pip install pytorch-lightning

In [27]:
df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42)
df_val, df_test = train_test_split(df_temp, test_size=0.25, random_state=42)

In [28]:
import pytorch_lightning as pl
from torch.utils.data import DataLoader, Dataset
import torch
import numpy as np

class TextDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.from_numpy(x).long()
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


class HotelReviewDataModule(pl.LightningDataModule):
    def __init__(self, df_train, df_val, df_test, max_len=100, batch_size=256):
        super().__init__()
        self.df_train = df_train
        self.df_val = df_val
        self.df_test = df_test
        self.max_len = max_len
        self.batch_size = batch_size

    def setup(self, stage=None):
        if stage == "fit" or stage is None:
            x_train = np.array([text_to_sequence(text, self.max_len)
                               for text in self.df_train["Review"]], dtype=np.int32)
            y_train = np.array(self.df_train["Rating"]) - 1

            x_val = np.array([text_to_sequence(text, self.max_len)
                             for text in self.df_val["Review"]], dtype=np.int32)
            y_val = np.array(self.df_val["Rating"]) - 1

            self.train_dataset = TextDataset(x_train, y_train)
            self.val_dataset = TextDataset(x_val, y_val)

        if stage == "test" or stage is None:
            x_test = np.array([text_to_sequence(text, self.max_len)
                              for text in self.df_test["Review"]], dtype=np.int32)
            y_test = np.array(self.df_test["Rating"]) - 1

            self.test_dataset = TextDataset(x_test, y_test)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size)

In [29]:
import pytorch_lightning as pl
import torch
import torch.nn as nn
from torchmetrics import Accuracy, F1Score

class ConvTextClassifierLightning(pl.LightningModule):
    def __init__(self, vocab_size=2000, embedding_dim=32, out_channel=128,
                 num_classes=5, learning_rate=0.001):
        super().__init__()
        self.save_hyperparameters()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.conv = nn.Conv1d(embedding_dim, out_channel, kernel_size=3)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.linear = nn.Linear(out_channel, num_classes)

        self.criterion = nn.CrossEntropyLoss()
        self.train_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.val_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.test_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.train_f1 = F1Score(task="multiclass", num_classes=num_classes)
        self.val_f1 = F1Score(task="multiclass", num_classes=num_classes)
        self.test_f1 = F1Score(task="multiclass", num_classes=num_classes)

    def forward(self, x):
        output = self.embedding(x)
        output = output.permute(0, 2, 1)
        output = self.conv(output)
        output = self.relu(output)
        output = torch.max(output, axis=2).values
        output = self.dropout(output)
        output = self.linear(output)
        return output

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)

        self.train_acc(preds, y)
        self.train_f1(preds, y)

        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", self.train_acc, prog_bar=True)
        self.log("train_f1", self.train_f1, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)

        self.val_acc(preds, y)
        self.val_f1(preds, y)

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", self.val_acc, prog_bar=True)
        self.log("val_f1", self.val_f1, prog_bar=True)

        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)

        self.test_acc(preds, y)
        self.test_f1(preds, y)

        self.log("test_loss", loss)
        self.log("test_acc", self.test_acc)
        self.log("test_f1", self.test_f1)

        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=2
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss"
            }
        }

In [30]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

data_module = HotelReviewDataModule(df_train, df_val, df_test, max_len=100, batch_size=256)

model = ConvTextClassifierLightning(
    vocab_size=10000,
    embedding_dim=64,
    out_channel=128,
    num_classes=5,
    learning_rate=0.001
)

checkpoint_callback = ModelCheckpoint(
    monitor='val_f1',
    mode='max',
    save_top_k=1,
    filename='best-{epoch:02d}-{val_f1:.4f}'
)

early_stop_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    mode='min'
)

trainer = pl.Trainer(
    max_epochs=20,
    callbacks=[checkpoint_callback, early_stop_callback],
    accelerator='gpu',
    log_every_n_steps=10
)

trainer.fit(model, data_module)
trainer.test(model, data_module)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name      ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ embedding │ Embedding          │  640 K │ train │     0 │
│ 1  │ conv      │ Conv1d             │ 24.7 K │ train │     0 │
│ 2  │ relu      │ ReLU               │      0 │ train │     0 │
│ 3  │ dropout   │ Dropout            │      0 │ train │     0 │
│ 4  │ linear    │ Linear             │    645 │ train │     0 │
│ 5  │ criterion │ CrossEntropyLoss   │      0 │ train │     0 │
│ 6  │ train_acc │ MulticlassAccuracy │      0 │ train │     0 │
│ 7  │ val_acc   │ MulticlassAccuracy │      0 │ train │     0 │
│ 8  │ test_acc  │ MulticlassAccuracy │      0 │ train │     0 │
│ 9  │ train_f1  │ MulticlassF1Score  │      0 │ train │     0 │
│ 10 │ val_f1    │ MulticlassF1Score  │      0 │ train │     0 │
│ 11 │ test_f1   │ MulticlassF1Score  │      0 │ train │     0 │
└────┴───────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 665 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 665 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 12                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=20` reached.


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.5795121788978577     │
│          test_f1          │    0.5795121788978577     │
│         test_loss         │    0.9412125945091248     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.9412125945091248,
  'test_acc': 0.5795121788978577,
  'test_f1': 0.5795121788978577}]

In [31]:
torch.cuda.empty_cache()

In [32]:
class LSTMTextClassifierLightning(ConvTextClassifierLightning):
    def __init__(self, vocab_size=5000, embedding_dim=64, hidden_size=128,
                 num_classes=5, learning_rate=0.001, num_layers=2):
        super().__init__(num_classes=num_classes, learning_rate=learning_rate)
        self.save_hyperparameters()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers=num_layers,
                           batch_first=True, dropout=0.3 if num_layers > 1 else 0)
        self.dropout = nn.Dropout(0.5)
        self.linear = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        output, (hidden, cell) = self.lstm(x)
        output = self.dropout(output[:, -1, :])
        output = self.linear(output)
        return output

In [33]:
lstm_model = LSTMTextClassifierLightning(vocab_size=10000, num_classes=5)


checkpoint_callback = ModelCheckpoint(
    monitor='val_f1',
    mode='max',
    save_top_k=1,
    filename='best-{epoch:02d}-{val_f1:.4f}'
)

early_stop_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    mode='min'
)

trainer = pl.Trainer(
    max_epochs=20,
    callbacks=[checkpoint_callback, early_stop_callback],
    accelerator='gpu',
    log_every_n_steps=10
)


trainer.fit(lstm_model, data_module)
results_lstm = trainer.test(lstm_model, data_module)

print("LSTM:", results_lstm)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name      ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ embedding │ Embedding          │  640 K │ train │     0 │
│ 1  │ conv      │ Conv1d             │ 12.4 K │ train │     0 │
│ 2  │ relu      │ ReLU               │      0 │ train │     0 │
│ 3  │ dropout   │ Dropout            │      0 │ train │     0 │
│ 4  │ linear    │ Linear             │    645 │ train │     0 │
│ 5  │ criterion │ CrossEntropyLoss   │      0 │ train │     0 │
│ 6  │ train_acc │ MulticlassAccuracy │      0 │ train │     0 │
│ 7  │ val_acc   │ MulticlassAccuracy │      0 │ train │     0 │
│ 8  │ test_acc  │ MulticlassAccuracy │      0 │ train │     0 │
│ 9  │ train_f1  │ MulticlassF1Score  │      0 │ train │     0 │
│ 10 │ val_f1    │ MulticlassF1Score  │      0 │ train │     0 │
│ 11 │ test_f1   │ MulticlassF1Score  │      0 │ train │     0 │
│ 12 │ lstm      │ LSTM               │  231 K │ train │     0 │
└────┴───────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 884 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 884 K                                                                                                
Total estimated model params size (MB): 3                                                                          
Modules in train mode: 13                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.5560975670814514     │
│          test_f1          │    0.5560975670814514     │
│         test_loss         │     1.073368787765503     │
└───────────────────────────┴───────────────────────────┘

LSTM: [{'test_loss': 1.073368787765503, 'test_acc': 0.5560975670814514, 'test_f1': 0.5560975670814514}]


In [34]:
class ImprovedCNN(ConvTextClassifierLightning):
    def __init__(self, vocab_size=10000, embedding_dim=128, out_channel=256,
                 num_classes=5, learning_rate=0.001):
        super().__init__()
        self.save_hyperparameters()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding_dropout = nn.Dropout(0.2)

        self.conv3 = nn.Conv1d(embedding_dim, out_channel, kernel_size=3, padding=1)
        self.conv4 = nn.Conv1d(embedding_dim, out_channel, kernel_size=4, padding=2)
        self.conv5 = nn.Conv1d(embedding_dim, out_channel, kernel_size=5, padding=2)

        self.bn1 = nn.BatchNorm1d(out_channel)
        self.bn2 = nn.BatchNorm1d(out_channel)
        self.bn3 = nn.BatchNorm1d(out_channel)

        self.dropout1 = nn.Dropout(0.5)

        self.fc1 = nn.Linear(out_channel * 3, 256)
        self.bn_fc = nn.BatchNorm1d(256)
        self.dropout2 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(256, num_classes)

        self.criterion = nn.CrossEntropyLoss()
        self.train_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.val_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.test_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.train_f1 = F1Score(task="multiclass", num_classes=num_classes)
        self.val_f1 = F1Score(task="multiclass", num_classes=num_classes)
        self.test_f1 = F1Score(task="multiclass", num_classes=num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        embedded = self.embedding_dropout(embedded)
        embedded = embedded.permute(0, 2, 1)

        conv3_out = torch.relu(self.bn1(self.conv3(embedded)))
        conv3_out = torch.max(conv3_out, dim=2)[0]

        conv4_out = torch.relu(self.bn2(self.conv4(embedded)))
        conv4_out = torch.max(conv4_out, dim=2)[0]

        conv5_out = torch.relu(self.bn3(self.conv5(embedded)))
        conv5_out = torch.max(conv5_out, dim=2)[0]

        concatenated = torch.cat([conv3_out, conv4_out, conv5_out], dim=1)
        concatenated = self.dropout1(concatenated)

        output = torch.relu(self.bn_fc(self.fc1(concatenated)))
        output = self.dropout2(output)
        output = self.fc2(output)

        return output

In [35]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

data_module = HotelReviewDataModule(df_train, df_val, df_test, max_len=100, batch_size=256)

model = ImprovedCNN(
    vocab_size=10000,
    embedding_dim=64,
    out_channel=128,
    num_classes=5,
    learning_rate=0.001
)

checkpoint_callback = ModelCheckpoint(
    monitor='val_f1',
    mode='max',
    save_top_k=1,
    filename='best-{epoch:02d}-{val_f1:.4f}'
)

early_stop_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    mode='min'
)

trainer = pl.Trainer(
    max_epochs=20,
    callbacks=[checkpoint_callback, early_stop_callback],
    accelerator='gpu',
    log_every_n_steps=10
)

trainer.fit(model, data_module)
trainer.test(model, data_module)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name              ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ embedding         │ Embedding          │  640 K │ train │     0 │
│ 1  │ conv              │ Conv1d             │ 12.4 K │ train │     0 │
│ 2  │ relu              │ ReLU               │      0 │ train │     0 │
│ 3  │ dropout           │ Dropout            │      0 │ train │     0 │
│ 4  │ linear            │ Linear             │    645 │ train │     0 │
│ 5  │ criterion         │ CrossEntropyLoss   │      0 │ train │     0 │
│ 6  │ train_acc         │ MulticlassAccuracy │      0 │ train │     0 │
│ 7  │ val_acc           │ MulticlassAccuracy │      0 │ train │     0 │
│ 8  │ test_acc          │ MulticlassAccuracy │      0 │ train │     0 │
│ 9  │ train_f1          │ MulticlassF1Score  │      0 │ train │     0 │
│ 10 │ val_f1            │ MulticlassF1Score  │      0 │ train │     0 │
│ 11 │ test_f1           │ MulticlassF1Score  │      0 │ train │     0 │
│ 12 │ embedding_dropout │ Dropout            │      0 │ train │     0 │
│ 13 │ conv3             │ Conv1d             │ 24.7 K │ train │     0 │
│ 14 │ conv4             │ Conv1d             │ 32.9 K │ train │     0 │
│ 15 │ conv5             │ Conv1d             │ 41.1 K │ train │     0 │
│ 16 │ bn1               │ BatchNorm1d        │    256 │ train │     0 │
│ 17 │ bn2               │ BatchNorm1d        │    256 │ train │     0 │
│ 18 │ bn3               │ BatchNorm1d        │    256 │ train │     0 │
│ 19 │ dropout1          │ Dropout            │      0 │ train │     0 │
│ 20 │ fc1               │ Linear             │ 98.6 K │ train │     0 │
│ 21 │ bn_fc             │ BatchNorm1d        │    512 │ train │     0 │
│ 22 │ dropout2          │ Dropout            │      0 │ train │     0 │
│ 23 │ fc2               │ Linear             │  1.3 K │ train │     0 │
└────┴───────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 852 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 852 K                                                                                                
Total estimated model params size (MB): 3                                                                          
Modules in train mode: 24                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=20` reached.


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.5434146523475647     │
│          test_f1          │    0.5434146523475647     │
│         test_loss         │    0.9957453608512878     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.9957453608512878,
  'test_acc': 0.5434146523475647,
  'test_f1': 0.5434146523475647}]